# 16 — Chọn feature cho model GIÁ CƠ BẢN (bộ TP.HCM)

Mục tiêu: chọn feature để dự đoán **giá cơ bản** = `target_shown_price / target_shown_multiplier`
(giá đã **bỏ surge**). Đây là Model A của Hướng 2 (giá cuối = giá cơ bản × hệ số nhân).

Khác với model giá cuối: giá cơ bản **không phụ thuộc surge** → kỳ vọng feature cung–cầu / giá quan
sát (đã gồm surge) sẽ **yếu**, còn quãng đường / thời lượng / dịch vụ **mạnh**. Notebook đưa cả 2 loại
feature vào để **5 phương pháp tự xác nhận** điều này.

Năm phương pháp: (1) Correlation & η · (2) Mutual Information · (3) Permutation importance ·
(4) RFE / backward selection · (5) SHAP. Cuối cùng đối chiếu → chốt bộ feature.

**Chuẩn bị: nạp dữ liệu, tạo target giá cơ bản, feature ứng viên**

In [1]:
import sys; sys.path.insert(0, ".")
from _common import *
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression, RFE
from sklearn.preprocessing import OrdinalEncoder
setup()
df = load(frac=0.12)   # doi frac=1.0 khi chay that

# Tao target gia co ban + feature base quan sat (da bo surge)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]
NUM = ["quote_distance", "quote_duration", "gio_vn", "thu_vn", "target_is_weekend", "latest_observed_base", "latest_observed_price", "history_60m_price_mean", "history_60m_price_std", "history_60m_price_min", "history_60m_price_max", "history_60m_price_slope_per_minute", "latest_observed_quote_distance", "latest_observed_quote_duration", "actual_observation_age_minutes", "pricing_market_imbalance_5m_lag", "pricing_demand_index_5m_lag", "weather_temp", "weather_humidity"]
FEATS = CAT + NUM
TARGET = "base_price"
K = 12
name_vn = "GIA CO BAN"

CAM = {"target_shown_price","target_shown_multiplier","target_price_per_km",
       "split","scenario_id","forecast_example_id","target_request_id","is_synthetic","base_price"}
assert not (set(FEATS) & CAM), "CO FEATURE LEAKAGE!"
print(f"Model GIA CO BAN: {len(FEATS)} feature ung vien ({len(CAT)} cat + {len(NUM)} num) | chon top K={K}")
print(f"Gia co ban: median {df.base_price.median():,.0f} VND (= gia / he so nhan)")

def prep(d, cat):
    X=d.copy()
    for c in cat: X[c]=X[c].astype("category")
    return X

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xrf = df[FEATS].copy()
Xrf[CAT] = enc.fit_transform(Xrf[CAT].astype(str))
Xrf = Xrf.fillna(Xrf.median(numeric_only=True))
y = df[TARGET].values

SELECTED = {}
def bang_chon(rank_series, ten):
    sel = list(rank_series.head(K).index)
    SELECTED[ten] = sel
    print(f"[{ten}] chon {len(sel)} feature:"); print("   " + ", ".join(sel))
    return rank_series.to_frame("diem")
print("San sang.")

Nap 827,693 dong (mau 12%) | gia median 114k VND | surge 81.7%
Model GIA CO BAN: 23 feature ung vien (4 cat + 19 num) | chon top K=12
Gia co ban: median 99,296 VND (= gia / he so nhan)


San sang.


#### 1. Correlation và Correlation ratio (η)

**Phương pháp là gì:** đo quan hệ trực tiếp giữa từng feature với target, không cần train model (nhóm *filter*).

**Cách thực hiện:** biến số dùng |Pearson correlation|; biến phân loại dùng η (0–1). Xếp hạng, chọn top K.

In [2]:
rows={}
for f in CAT: rows[f]=eta(df[f], df[TARGET])
for f in NUM: rows[f]=abs(df[f].corr(df[TARGET]))
rank_corr = pd.Series(rows).sort_values(ascending=False)
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_corr, "Correlation/eta").round(4))

Ket qua — xep hang + feature chon:
[Correlation/eta] chon 12 feature:
   quote_distance, quote_duration, history_60m_price_mean, latest_observed_quote_distance, history_60m_price_min, latest_observed_base, latest_observed_price, history_60m_price_max, history_60m_price_std, latest_observed_quote_duration, pickup_location_name, dropoff_location_name


,diem
quote_distance,0.7640
quote_duration,0.6994
history_60m_price_mean,0.4988
latest_observed_quote_distance,0.4710
history_60m_price_min,0.4561
latest_observed_base,0.3687
latest_observed_price,0.3632
history_60m_price_max,0.3459
history_60m_price_std,0.3397
latest_observed_quote_duration,0.3370


#### 2. Mutual Information

**Phương pháp là gì:** đo lượng thông tin feature cung cấp về target — bắt cả quan hệ phi tuyến (nhóm *filter*).

**Cách thực hiện:** tính MI của từng feature (đã mã hoá số) với target trên một mẫu; xếp hạng, chọn top K.

In [3]:
samp = np.random.RandomState(0).choice(len(Xrf), min(40000,len(Xrf)), replace=False)
mi = mutual_info_regression(Xrf.iloc[samp], y[samp], random_state=0)
rank_mi = pd.Series(mi, index=FEATS).sort_values(ascending=False)
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_mi, "MutualInfo").round(4))

Ket qua — xep hang + feature chon:
[MutualInfo] chon 12 feature:
   quote_distance, quote_duration, history_60m_price_mean, latest_observed_quote_distance, history_60m_price_min, latest_observed_base, history_60m_price_max, latest_observed_price, gio_vn, history_60m_price_std, latest_observed_quote_duration, pickup_location_name


,diem
quote_distance,0.4926
quote_duration,0.3789
history_60m_price_mean,0.2437
latest_observed_quote_distance,0.2414
history_60m_price_min,0.2125
latest_observed_base,0.1664
history_60m_price_max,0.1612
latest_observed_price,0.1439
gio_vn,0.1420
history_60m_price_std,0.1256


#### 3. Permutation importance

**Phương pháp là gì:** đo feature mà model thực sự dùng — xáo trộn 1 feature rồi xem sai số tăng (nhóm *embedded*).

**Cách thực hiện:** train HistGradientBoosting trên tập train; trên validation, xáo trộn từng feature; xếp hạng, chọn top K.

In [4]:
tr = df[df.split=="train"].sample(min(60000,(df.split=="train").sum()), random_state=1)
va = df[df.split=="validation"].sample(min(20000,(df.split=="validation").sum()), random_state=2)
m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, categorical_features=CAT,
    random_state=42).fit(prep(tr,CAT)[FEATS], np.log(tr[TARGET]))
pi = permutation_importance(m, prep(va,CAT)[FEATS], np.log(va[TARGET]), n_repeats=3, random_state=0, n_jobs=1)
rank_perm = pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False)
print("Ket qua — xep hang + feature chon:")
display(bang_chon(rank_perm, "Permutation").round(4))

Ket qua — xep hang + feature chon:
[Permutation] chon 12 feature:
   quote_distance, quote_duration, service_name, pickup_location_name, pricing_market_imbalance_5m_lag, history_60m_price_min, latest_observed_quote_distance, pricing_demand_index_5m_lag, latest_observed_base, history_60m_price_slope_per_minute, weather_temp, weather_humidity


,diem
quote_distance,0.7273
quote_duration,0.2065
service_name,0.0101
pickup_location_name,0.0015
pricing_market_imbalance_5m_lag,0.0006
history_60m_price_min,0.0002
latest_observed_quote_distance,0.0002
pricing_demand_index_5m_lag,0.0002
latest_observed_base,0.0001
history_60m_price_slope_per_minute,0.0001


#### 4. RFE — Recursive Feature Elimination (backward selection)

**Phương pháp là gì:** wrapper — bắt đầu với tất cả feature, loại dần feature yếu nhất, train lại, đến khi còn K.

**Cách thực hiện:** dùng RandomForest (có `feature_importances_`); RFE loại 2 feature mỗi vòng cho tới khi còn K.

In [5]:
s = np.random.RandomState(3).choice(len(Xrf), min(25000,len(Xrf)), replace=False)
rf = RandomForestRegressor(n_estimators=60, max_depth=14, n_jobs=-1, random_state=42)
rfe = RFE(rf, n_features_to_select=K, step=2).fit(Xrf.iloc[s], y[s])
rank_rfe = pd.Series(rfe.ranking_, index=FEATS).sort_values()
SELECTED["RFE"] = [f for f,keep in zip(FEATS, rfe.support_) if keep]
print(f"[RFE] chon {len(SELECTED['RFE'])} feature:"); print("   " + ", ".join(SELECTED["RFE"]))
print("\n(ranking: 1 = duoc chon)")
display(rank_rfe.to_frame("ranking"))

[RFE] chon 12 feature:
   quote_distance, quote_duration, latest_observed_base, history_60m_price_std, history_60m_price_slope_per_minute, latest_observed_quote_distance, latest_observed_quote_duration, actual_observation_age_minutes, pricing_market_imbalance_5m_lag, pricing_demand_index_5m_lag, weather_temp, weather_humidity

(ranking: 1 = duoc chon)


,ranking
quote_duration,1
quote_distance,1
history_60m_price_std,1
latest_observed_base,1
history_60m_price_slope_per_minute,1
actual_observation_age_minutes,1
latest_observed_quote_duration,1
latest_observed_quote_distance,1
pricing_market_imbalance_5m_lag,1
weather_humidity,1


#### 5. SHAP

**Phương pháp là gì:** đo đóng góp từng feature vào từng dự đoán (Shapley), lấy mean|đóng góp| làm mức quan trọng (nhóm *embedded*).

**Cách thực hiện:** train RandomForest, dùng `TreeExplainer` tính SHAP trên mẫu nhỏ; mean(|SHAP|); xếp hạng, chọn top K.

In [6]:
try:
    import shap
    s2 = np.random.RandomState(5).choice(len(Xrf), min(20000,len(Xrf)), replace=False)
    rf2 = RandomForestRegressor(n_estimators=60, max_depth=14, n_jobs=-1, random_state=42).fit(Xrf.iloc[s2], y[s2])
    Xs = Xrf.iloc[np.random.RandomState(6).choice(len(Xrf), 1500, replace=False)]
    sv = shap.TreeExplainer(rf2).shap_values(Xs)
    rank_shap = pd.Series(np.abs(sv).mean(0), index=FEATS).sort_values(ascending=False)
    print("Ket qua — xep hang + feature chon:")
    display(bang_chon(rank_shap, "SHAP").round(4))
except ImportError:
    print("Chua cai shap. Chay: pip install shap"); SELECTED["SHAP"]=[]

Ket qua — xep hang + feature chon:
[SHAP] chon 12 feature:
   quote_distance, quote_duration, latest_observed_quote_distance, pricing_demand_index_5m_lag, actual_observation_age_minutes, history_60m_price_slope_per_minute, weather_humidity, history_60m_price_std, latest_observed_quote_duration, latest_observed_base, pricing_market_imbalance_5m_lag, history_60m_price_max


,diem
quote_distance,17084.4153
quote_duration,8320.9779
latest_observed_quote_distance,330.1378
pricing_demand_index_5m_lag,301.0788
actual_observation_age_minutes,286.1228
history_60m_price_slope_per_minute,276.1313
weather_humidity,266.6338
history_60m_price_std,257.8658
latest_observed_quote_duration,251.2247
latest_observed_base,248.1869


#### Kiểm tra trùng lặp (bổ trợ)

Hai feature |r| > 0,9 mang thông tin trùng → bỏ bớt 1.

In [7]:
import itertools
corr = df[NUM].corr().abs()
trung=[(a,b,round(corr.loc[a,b],3)) for a,b in itertools.combinations(NUM,2) if corr.loc[a,b]>0.9]
if trung:
    print("Cap TRUNG LAP (|r|>0.9):")
    for a,b,r in sorted(trung,key=lambda x:-x[2]): print(f"   {a:34} <-> {b:34} r={r}")
else:
    print("Khong co cap |r|>0.9.")

Cap TRUNG LAP (|r|>0.9):
   history_60m_price_std              <-> history_60m_price_max              r=0.929
   weather_temp                       <-> weather_humidity                   r=0.924
   history_60m_price_mean             <-> history_60m_price_min              r=0.916
   latest_observed_base               <-> latest_observed_price              r=0.914


#### Đối chiếu 5 phương pháp và chốt bộ feature

Đếm mỗi feature được bao nhiêu phương pháp chọn. Bộ chốt = feature được ≥ 3/5 phương pháp đồng thuận.

In [8]:
pps = ["Correlation/eta","MutualInfo","Permutation","RFE","SHAP"]
dc = pd.DataFrame(0, index=FEATS, columns=pps)
for pp in pps:
    for f in SELECTED.get(pp,[]): dc.loc[f,pp]=1
dc["so_pp_chon"] = dc.sum(axis=1)
dc = dc.sort_values("so_pp_chon", ascending=False)
print("BANG DOI CHIEU 5 PHUONG PHAP (1 = duoc chon):")
display(dc)

BANG DOI CHIEU 5 PHUONG PHAP (1 = duoc chon):


,Correlation/eta,MutualInfo,Permutation,RFE,SHAP,so_pp_chon
quote_distance,1,1,1,1,1,5
quote_duration,1,1,1,1,1,5
latest_observed_base,1,1,1,1,1,5
latest_observed_quote_distance,1,1,1,1,1,5
latest_observed_quote_duration,1,1,0,1,1,4
history_60m_price_std,1,1,0,1,1,4
pickup_location_name,1,1,1,0,0,3
weather_humidity,0,0,1,1,1,3
pricing_demand_index_5m_lag,0,0,1,1,1,3
history_60m_price_min,1,1,1,0,0,3


In [9]:
chot = list(dc[dc.so_pp_chon >= 3].index)
chot_cat = [f for f in chot if f in CAT]; chot_num = [f for f in chot if f in NUM]
print("="*60)
print(f"BO FEATURE CHOT — MODEL {name_vn} (>= 3/5 phuong phap)")
print("="*60)
print(f"  Categorical ({len(chot_cat)}): {chot_cat}")
print(f"  Numeric     ({len(chot_num)}): {chot_num}")
print(f"  Tong: {len(chot)} feature  |  Target: base_price")
print("\n  Ky vong: quang duong, thoi luong, dich vu manh; cung-cau/gia-quan-sat yeu (base bo surge).")

BO FEATURE CHOT — MODEL GIA CO BAN (>= 3/5 phuong phap)
  Categorical (1): ['pickup_location_name']
  Numeric     (12): ['quote_distance', 'quote_duration', 'latest_observed_base', 'latest_observed_quote_distance', 'latest_observed_quote_duration', 'history_60m_price_std', 'weather_humidity', 'pricing_demand_index_5m_lag', 'history_60m_price_min', 'history_60m_price_max', 'pricing_market_imbalance_5m_lag', 'history_60m_price_slope_per_minute']
  Tong: 13 feature  |  Target: base_price

  Ky vong: quang duong, thoi luong, dich vu manh; cung-cau/gia-quan-sat yeu (base bo surge).
